In [ ]:
!pip3 install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import numpy as np

# fetch dataset
wine_quality = fetch_ucirepo(id=186)

# data (as pandas dataframes)
X = wine_quality.data.features
numeric_data = X.to_numpy(dtype=float)
y = wine_quality.data.targets
categorical_data = wine_quality.data.original
fulldf = categorical_data[['color']]
cat = fulldf.to_numpy(object)


# metadata
#print(wine_quality.metadata)

# variable information
#print(wine_quality.variables)
print(cat)

## Helper Functions ##

In [ ]:
def _is_nan(x):
    # works for python floats and numpy float types
    return x != x

In [ ]:
def _safe_float(x):
    # convert to float if possible;
    try:
        return float(x)
    except:
        return x

In [ ]:
def _col_as_list(data, j):
    n = data.shape[0]
    col = [0.0] * n
    for i in range(n):
        col[i] = data[i, j]
    return col

In [ ]:
def _sum_1d(arr):
    s = 0.0
    for v in arr:
        s += v
    return s

In [ ]:
def _mean_1d(arr):
    n = len(arr)
    if n == 0:
        raise ValueError("Cannot compute mean of empty array")
    return _sum_1d(arr) / n

In [ ]:

from IPython.display import display, Markdown

# Basic dataset size
n_rows, n_cols = X.shape

# Numeric vs categorical columns
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(exclude=[np.number]).columns

n_numeric = len(num_cols)
n_categorical = len(cat_cols)

# Store column names as lists
num_cols_list = list(num_cols)
cat_cols_list = list(cat_cols)


In [ ]:
from IPython.display import display, Markdown

missing_counts = X.isna().sum()
total_missing_cells = int(missing_counts.sum())
any_missing = bool(X.isna().any().any())

In [ ]:
num_df = X.select_dtypes(include=[np.number])
quality_corr = corr_matrix["quality"].drop(labels=["quality"])

# sort by correlation
quality_corr_sorted = quality_corr.reindex(quality_corr.abs().sort_values(ascending=False).index)

#select top attributes
k = 5
top_k = quality_corr_sorted.head(k)

# Format for presentation
top_lines = []
for attr, corr_val in top_k.items():
    sign = "+" if corr_val >= 0 else "-"
    top_lines.append(f"- **{attr}**: {sign}{abs(corr_val):.3f}")

top_text = "\n".join(top_lines)

## Problem 1

a) What is the data used for (explain this in a well-written paragraph)? [5 points]

This dataset uses Physiochemical measurements and expert quality scores to understand wine samples. It helps study how measurable chemical properties like acidity, sugar, sulphate, and alcohol content are related to perceived quality. Because these are very quantifiable qualities; this allows for easy determiniation of data analysis dynamics for modeling.

b) Who (or what organization) uploaded the data? [1 point]

Science Direct posted it, but the creator is Paulo Cortez, Antonio Cerdeira, Fernando Almedia, Telmo Matos, and Jose Reis from the department of Information Systems / R&D Centre Algoritmi, University of Minho and Viticulture Commission of the Vinho Verde Region. 

c) How many attributes and how many entries are included in the data?
a. How many numerical attributes? [1 point]
**{n_numeric} numerical attributes**
b. How many categorical attributes? [1 point]
**{n_categorical} categorical attribute(s)**
c. For the categorical attributes, would you suggest integer encoding (label encoding) or
one-hot encoding? And why? Answer this for each categorical attribute. [3 points]
 The dataset contains **{n_cols} attributes** with **{n_rows} instances**.  
There are **{n_numeric} numerical attributes** and **{n_categorical} categorical attribute(s)**.

{encoding_text}
"""))

d) Missing values:

a. Are there missing values in this dataset? [1 point]

Total missing cells in the dataset: **{total_missing_cells}**  
Any missing values present?: **{any_missing}**

b. If so, for each attribute, what is the proportion of the data that is missing? [4 points]

{"No missing values were found, so no imputation was required." if not any_missing
else "Missing values were detected, so imputation or removal is required before analysis."}
"""))

c. What is the proportion of the missing data overall? (You can use the plots to summarize
this data and explain each answer). [1 point]

None

e) What is the most interesting thing about this dataset to you? Explain this in a well-written
paragraph. [6 points]

The dataset contains numerical and categorical data. Continuous physiochemical measurements of each sample, listed above, make up a full set of numerical attributes. Additionally, the categorical separation of red and white wines forms a complete and informatiove dataset. It educates the difference in qualities provided in certain grape colros; and the qualitative expectation of thses categories.

f) Out of all the attributes in this dataset, what do you think are the most descriptive attributes?
(before doing any data analysis)? [5 points]

The top {k} most descriptive numerical attributes (ranked by absolute correlation with quality) are:

{top_text}

These attributes show the strongest linear relationships with the quality score, and so are more helpful for explaining or predicting wine quality in this dataset.


## Problem 2 
a) Write a function that computes the multi-dimensional mean of the numerical attributes. You should return the answer as a 2D NumPy array. [5 points]


In [ ]:
# returns 2D numpy array: (1 x d)
def multidimensional_mean(data_2d):
    n, d = data_2d.shape
    means = np.zeros((1, d), dtype=float)

    for j in range(d):
        s = 0.0
        for i in range(n):
            s += float(data_2d[i, j])
        means[0, j] = s / n

    return means

b) A function to calculate sample variance of a given attribute. [5 points]

In [ ]:
# s^2 = (1/(n-1)) * sum (xi - xbar)^2
def sample_variance(attribute_1d):
    n = len(attribute_1d)
    if n < 2:
        raise ValueError("Sample variance requires n >= 2")

    # mean manually
    s = 0.0
    for i in range(n):
        s += float(attribute_1d[i])
    mean = s / n

    # squared deviations
    ss = 0.0
    for i in range(n):
        diff = float(attribute_1d[i]) - mean
        ss += diff * diff

    return ss / (n - 1)

c) A function to calculate sample covariance between two given attributes. [5 points]

In [ ]:
# cov(X,Y) = (1/(n-1))*sum (xi-xbar)(yi-ybar)
def sample_covariance(attr1_1d, attr2_1d):
    n = len(attr1_1d)
    if n != len(attr2_1d):
        raise ValueError("Attributes must have same length")
    if n < 2:
        raise ValueError("Sample covariance requires n >= 2")

    # means manually
    s1 = 0.0
    s2 = 0.0
    for i in range(n):
        s1 += float(attr1_1d[i])
        s2 += float(attr2_1d[i])
    m1 = s1 / n
    m2 = s2 / n

    # covariance sum
    sxy = 0.0
    for i in range(n):
        sxy += (float(attr1_1d[i]) - m1) * (float(attr2_1d[i]) - m2)

    return sxy / (n - 1)

d) A function that calculates the covariance matrix. (You can use the functions that you have
created in b and c). [5 points]


In [ ]:
# returns d x d numpy array
def covariance_matrix(data_2d):
    n, d = data_2d.shape
    cov = np.zeros((d, d), dtype=float)

    # pre-extract columns to python lists
    cols = []
    for j in range(d):
        cols.append(_col_as_list(data_2d, j))

    for i in range(d):
        for j in range(d):
            cov[i, j] = sample_covariance(cols[i], cols[j])

    return cov

e) A function to calculate the correlation coefficient between two attributes. [5 points]


In [ ]:
# corr = cov / (std1 * std2), std = sqrt(variance)
def correlation_coefficient(attr1_1d, attr2_1d):
    cov = sample_covariance(attr1_1d, attr2_1d)
    var1 = sample_variance(attr1_1d)
    var2 = sample_variance(attr2_1d)

    if var1 == 0.0 or var2 == 0.0:
        raise ValueError("Correlation undefined when variance is zero")

    std1 = var1 ** 0.5
    std2 = var2 ** 0.5
    return cov / (std1 * std2)

f) A function that normalizes the attributes in a 2D NumPy array using z-score normalization. [5
points]


In [ ]:
# z = (x - mean) / std  (use sample std with ddof=1)
def z_score_normalization(data_2d):
    n, d = data_2d.shape
    out = np.zeros((n, d), dtype=float)

    # compute per-column mean and sample std manually
    means = [0.0] * d
    stds  = [0.0] * d

    for j in range(d):
        # mean
        s = 0.0
        for i in range(n):
            s += float(data_2d[i, j])
        means[j] = s / n

        # sample variance
        ss = 0.0
        for i in range(n):
            diff = float(data_2d[i, j]) - means[j]
            ss += diff * diff
        if n < 2:
            raise ValueError("Need n >= 2 for sample std")
        var = ss / (n - 1)
        stds[j] = var ** 0.5

        if stds[j] == 0.0:
            # if constant column, z-score would divide by zero
            # you can set to 0, or raise error. I set to 0.
            stds[j] = 0.0

    # normalize
    for i in range(n):
        for j in range(d):
            if stds[j] == 0.0:
                out[i, j] = 0.0
            else:
                out[i, j] = (float(data_2d[i, j]) - means[j]) / stds[j]

    return out

g) A function that normalizes the attributes in a 2D NumPy array using range normalization. [5
points]


In [ ]:
# r = (x - min) / (max - min)
def range_normalization(data_2d):
    n, d = data_2d.shape
    out = np.zeros((n, d), dtype=float)

    mins = [0.0] * d
    maxs = [0.0] * d

    for j in range(d):
        # initialize min/max from first element
        mn = float(data_2d[0, j])
        mx = float(data_2d[0, j])

        for i in range(1, n):
            v = float(data_2d[i, j])
            if v < mn:
                mn = v
            if v > mx:
                mx = v

        mins[j] = mn
        maxs[j] = mx

    for i in range(n):
        for j in range(d):
            denom = (maxs[j] - mins[j])
            if denom == 0.0:
                out[i, j] = 0.0
            else:
                out[i, j] = (float(data_2d[i, j]) - mins[j]) / denom

    return out

h) A function that will compute the covariance matrix of a dataset. [5 points]

In [ ]:
# (This is essentially (d) again;
def compute_covariance_matrix(data_2d):
    return covariance_matrix(data_2d)

i) A function that will label-encode a 2D categorical data array that is passed as an input. The
function can expect that the user is passing a 2D array with attributes that are only categorical,
and the function should output a new data matrix with the converted label-encoded (integerencoded) data. [5 points]

In [ ]:
# returns int numpy array
def label_encode_2d(cat_2d):
    n, d = cat_2d.shape
    out = np.zeros((n, d), dtype=int)

    for j in range(d):
        mapping = {}      # category -> int
        next_id = 0

        for i in range(n):
            key = cat_2d[i, j]
            # convert numpy scalar to python type for dict keys
            try:
                key = key.item()
            except:
                pass

            if key not in mapping:
                mapping[key] = next_id
                next_id += 1

            out[i, j] = mapping[key]

    return out

# Problem 3

**(A)** Calculate multi-dimensional mean and covariance matrix of the numerical portion of
the data.

In [ ]:
# Assume numeric_data is a 2D numpy array of only numerical attributes

mean_vector = multidimensional_mean(numeric_data)
cov_matrix = covariance_matrix(numeric_data)

print("Multi-dimensional Mean:")
print(mean_vector)

print("\nCovariance Matrix:")
print(cov_matrix)


**(B)** Convert all the categorical data attributes to numerical attributes using label encoding or one-hot encoding.

In [ ]:
encoded_categorical = label_encode_2d(cat)

print("Encoded Categorical Data:")
print(encoded_categorical)


**(C)** If your data has missing values, fill these values with attribute mean.

In [ ]:
numeric_df = X.select_dtypes(include=[np.number])
numeric_data = numeric_df.to_numpy(dtype=float)

# Categorical data (n x q)
cat_df = X.select_dtypes(exclude=[np.number])
cat_data = cat_df.to_numpy(object)

print("numeric_data shape:", numeric_data.shape)
print("cat_data shape:", cat_data.shape)

**(D)** Recalculate the multi-dimensional mean of the data matrix after the transformation (categorical
attributes are now numerical).

In [ ]:
# Build transformed matrix: numeric attributes + encoded categorical attributes
num = np.array(numeric_data, dtype=float)

# Fill missing numeric values with column mean
for j in range(num.shape[1]):
    col = num[:, j]
    m = np.nanmean(col)
    mask = np.isnan(col)
    if np.any(mask):
        num[mask, j] = m

# Use encoded categorical data from part (B)
cat_num = np.array(encoded_categorical, dtype=float)

transformed_data = np.hstack((num, cat_num))

new_mean = multidimensional_mean(transformed_data)
print("New Multi-dimensional Mean (after transformation):")
print(new_mean)


**(E)** What is the new covariance matrix of the data (categorical attributes are now numerical).

In [ ]:
new_cov = covariance_matrix(transformed_data)
print("New Covariance Matrix (after transformation):")
print(new_cov)


**(F)** Choose 4 pairs of numerical attributes that you think might be related. Create scatter plots for
these 4 pairs of attributes, along with the description and analysis of why these pairs of
attributes are related and how the scatter plot does or does not support your intuition.

In [ ]:
!pip3 install matplotlib
!pip3 install pandas as pd

In [ ]:

import matplotlib.pyplot as plt

print(X)

pairs = [("alcohol", "quality"), ("residual_sugar", "density"), ("pH", "fixed_acidity"), ("pH", "quality")]

numeric_cols = X.select_dtypes(include=['number']).columns
valid_pairs = [(a, b) for a, b in pairs if a in numeric_cols and b in numeric_cols]

print(list(valid_pairs))
for (x_cols, y_cols) in valid_pairs: 
    df_xy = X[[x_cols, y_cols]].dropna()
    x = df_xy[x_cols]
    y = df_xy[y_cols]

    plt.figure(figsize=(8, 6))
    plt.scatter(x, y, alpha=.5)
    plt.title(f"{x_cols} vs {y_cols}")
    plt.xlabel(x_cols)
    plt.show()

**(G)** Which range-normalized numerical attributes have the largest estimated covariance? What is
the covariance? Create a scatter plot of these range-normalized attributes.

In [ ]:
numeric_data = X.select_dtypes(include=['number'])

normalized = (numeric_data - numeric_data.min()) / (numeric_data.max() - numeric_data.min())
covariance_mat = normalized.cov()
print("Covariance Matrix")
print(covariance_mat)

sorted = covariance_matrix.unstack().sort_values(ascending=False).drop_duplicates()
# not sure why it does that again abut it gets the same attributes corvariance as the first 10 values, so i just get the last one
largest = sorted.head(10)

attribute1, attribute2 = largest.tail(1).index[0]
print(f"Highest covariance is between {attribute1} and {attribute2}")
largest_cov = sorted.tail(1).values[0]
print(f"Covariance value: {largest_cov}")

plt.figure(figsize=(8, 6))
plt.scatter(normalized[attribute1], normalized[attribute2], alpha=.5)
plt.title(f"{attribute1} vs {attribute2} (normalized)")
plt.xlabel(attribute1)  
plt.ylabel(attribute2)
plt.show()

**(H)** Which z-score normalized numerical attributes have the largest estimated correlation
coefficient between them? What is that value? Create a scatter plot for these two z-score
normalized attributes.

In [ ]:
z_scored = (numeric_data - numeric_data.mean()) / numeric_data.std(ddof=1)
correlatoin_matrix = z_scored.corr()
#print("Correlation Matrix with z-score normalization")
#print(correlatoin_matrix)

sorted_corr = correlatoin_matrix.unstack().sort_values(ascending=False).drop_duplicates()
# again what the hell
largest_corr = sorted_corr.head(10)
attribute1_corr, attribute2_corr = largest_corr.tail(1).index[0]
value = correlatoin_matrix.loc[attribute1_corr, attribute2_corr]
print(f"Highest correlation is between {attribute1_corr} and {attribute2_corr}")    
print(f"Correlation value: {value}")

plt.figure(figsize=(8, 6))
plt.scatter(z_scored[attribute1_corr], z_scored[attribute2_corr], alpha=.5)
plt.title(f"{attribute1_corr} vs {attribute2_corr} (z-score normalized)")
plt.xlabel(attribute1_corr)  
plt.ylabel(attribute2_corr)
plt.show()

**(I)** Which z-score normalized numerical attributes have the smallest estimated correlatoin coefficient between them? What is that value? Create a scatter plot for these two z-score normalized attributes.

In [ ]:
smallest = sorted_corr.tail(1)
attribute1_small, attribute2_small = smallest.index[0]
print(f"Lowest correlation is between {attribute1_small} and {attribute2_small}")
smallest_value = sorted_corr.tail(1).values[0] 
print(f"Correlation value: {smallest_value}")
plt.figure(figsize=(8, 6))
plt.scatter(z_scored[attribute1_small], z_scored[attribute2_small], alpha=.5)
plt.title(f"{attribute1_small} vs {attribute2_small} (z-score normalized)")
plt.xlabel(attribute1_small)
plt.ylabel(attribute2_small)
plt.show()

**(J)** How many pairs of numerical features have a correlation greater than or equal to 0.5?

In [ ]:
#VS Code AI helped me with this one ngl

n, d = numeric_data[0].
num_pairs = 0
pairs = []
for i in range(d):
    col_i = numeric_data.iloc[:, i]
    for j in range(i + 1, d):
        col_j = numeric_data.iloc[:, j]
        corr = correlation_coefficient(col_i, col_j)
        if corr >= 0.5:
            num_pairs += 1
            pairs.append((numeric_data.columns[i], numeric_data.columns[j], corr))
print(f"Number of attribute pairs with correlation >= 0.5: {num_pairs}")


#upper_pairs = correlatoin_matrix.where(np.triu(np.ones(correlatoin_matrix.shape), k=1).astype(bool)).unstack().dropna()
#print(upper_pairs.sort_values(ascending=False))
#count = 0
#for corr in upper_pairs.sort_values(ascending=False):
    #if corr >= .5:
        #count += 1

#print(f"Number of attribute pairs with correlation >= 0.5: {count}")


**(K)** How many pairs of numerical features have negative estimated covariance? 

In [ ]:
lower_pairs = correlatoin_matrix.where(np.tril(np.ones(correlatoin_matrix.shape), k=-1).astype(bool)).unstack().dropna()
print(lower_pairs.sort_values())
count = 0
for corr in lower_pairs.sort_values():
    if corr < 0.0:
        count += 1
print(f"Number of attribute with negative correlation: {count}")


**(L)** Calculate the total variance of the data

In [ ]:
names = list(X.columns)
numeric_data = numeric_df.values.tolist()
print(cov_matrix)


n = len(names)
variance = 0.0
for i in range(n):
    variance += cov_matrix[i, i]
print(f"Total variance (sum of diagonal of covariance matrix): {variance}")

**(M)**  What is the ratio of the total variance of the data restricted to the five features with the highest
estimated variance to the total estimated variance of the data?

In [ ]:
def list_column(data, index):
    return [row[index] for row in data]

figures = list(numeric_df.columns)
figure_array = numeric_df.values.tolist()

i = len(names)
variance = []
for l in range(i):
    v = sample_variance(list_column(figure_array, l))
    variance.append(v)

top5 = []
for variable in variance:
    if len(top5) < 5:
        top5.append(variable)
    else:
        min_top5 = min(top5)
        if variable > min_top5:
            top5.remove(min_top5)
            top5.append(variable)
print("Top 5 attributes with highest variance:")
for var in top5:
    index = variance.index(var)
    print(f"{names[index]}: {var}")

